# Rank LoRA untuk Klasifikasi Sentimen Bahasa Indonesia pada Model Kecil Terkuantisasi

Notebook ini menjalankan seluruh eksperimen dalam satu kali duduk: menyiapkan data SmSA,
melatih empat nilai rank adaptor QLoRA dengan tiga seed, memilih rank memakai data validasi,
lalu mengevaluasi data uji satu kali.

**Kebutuhan:** Google Colab dengan GPU T4 (memori 15 GB). Tanpa GPU, jangan dijalankan.

**Yang disimpan:** semua hasil tersimpan di Google Drive pada folder `MyDrive/qwen25-05b-qlora-smsa/`,
sehingga sesi yang terputus bisa dilanjutkan tanpa mengulang run yang sudah selesai.

| Folder | Isi |
|---|---|
| `configs/` | `config.json`, `environment.json`, `urutan_eksekusi.json` |
| `data/processed/` | `train.csv`, `valid.csv`, `test.csv`, `laporan_duplikat.json` |
| `runs/` | per run: `hasil.json`, `pred_valid.csv`, `adaptor/`, `kurva_loss.csv` |
| `results/` | tabel agregat, keputusan rank, hasil uji, gambar |
| `logs/` | catatan stdout dan cek tokenizer |
| `notes/` | changelog dan catatan proses |

**Cara pakai:** jalankan sel berurutan dari atas. Bila sesi terputus, sambungkan ulang
Runtime, lalu jalankan lagi sel 1 sampai 4, kemudian langsung ke sel 10 (grid pelatihan).
Run yang sudah selesai otomatis dilewati.

**Artefak yang dikirim balik:** sel terakhir membuat `hasil-colab.zip` di Drive.
Unggah berkas itu untuk diolah menjadi laporan.

In [ ]:
#@title 1) Periksa perangkat
import platform, subprocess, sys, os

print("python :", sys.version.split()[0], "|", platform.platform())
try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    print("gpu    :", out if out else "tidak terdeteksi")
except Exception as e:
    print("gpu    : nvidia-smi tidak tersedia", e)

import torch
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu torch:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"vram   : {free/1024**2:.0f} MiB bebas dari {total/1024**2:.0f} MiB")
else:
    raise SystemExit("GPU tidak aktif. Pilih Runtime > Change runtime type > T4 GPU.")

In [ ]:
#@title 2) Pasang paket
%pip install -q -U transformers peft bitsandbytes accelerate pandas scikit-learn matplotlib

import importlib.metadata as md
for p in ["torch", "transformers", "peft", "bitsandbytes", "accelerate", "pandas",
          "numpy", "scikit-learn", "matplotlib"]:
    try:
        print(f"{p:14s} {md.version(p)}")
    except Exception:
        print(f"{p:14s} tidak terpasang")

In [ ]:
#@title 3) Sambungkan Drive dan siapkan folder kerja
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/qwen25-05b-qlora-smsa")
for sub in ["configs", "data/raw", "data/processed", "runs", "results", "logs", "notes"]:
    (BASE / sub).mkdir(parents=True, exist_ok=True)

# simpan cache model di Drive supaya tidak mengunduh ulang setelah sesi terputus
os.environ["HF_HOME"] = str(BASE / "hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("folder kerja:", BASE)
print("cache model :", os.environ["HF_HOME"])

In [ ]:
#@title 4) Konfigurasi penelitian (nilai dibekukan, sama dengan naskah)
import json, os

CONFIG = {
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "seeds": [42, 123, 2026],
    "ranks": [4, 8, 16, 32],
    "alpha_multiplier": 2,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "v_proj"],
    "bias": "none",
    "lr": 0.0002,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "epochs": 3,
    "warmup_ratio": 0.03,
    "microbatch": 1,
    "grad_accum": 16,
    "max_len": 256,
    "eval_batch": 4,
    "n_train_subset": 2000,
    "subset_seed": 2026,
    "schedule_seed": 2026,
    "tolerance": 0.01,
    "bootstrap_reps": 2000,
    "bootstrap_seed": 2026,
    "pilot_steps": 20,
    "label_map": {"negative": 0, "neutral": 1, "positive": 2},
    "prompt": ("Klasifikasikan sentimen teks berikut. Jawab dengan satu kode: "
               "A untuk negatif, B untuk netral, C untuk positif.\nTeks: {teks}\nJawaban:"),
    "data_base_url": "https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa",
    "data_files": {"train": "train_preprocess.tsv", "valid": "valid_preprocess.tsv",
                   "test": "test_preprocess.tsv"},
}

(BASE / "configs" / "config.json").write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")

import importlib.metadata as md, platform, sys
def ver(p):
    try:
        return md.version(p)
    except Exception:
        return None

env = {
    "platform": platform.platform(),
    "python": sys.version.split()[0],
    "torch": ver("torch"),
    "cuda": None,
    "gpu": None,
    "vram_total_mib": None,
    "packages": {p: ver(p) for p in ["transformers", "peft", "bitsandbytes", "accelerate",
                                    "pandas", "numpy", "scikit-learn", "matplotlib"]},
}
import torch
if torch.cuda.is_available():
    env["cuda"] = torch.version.cuda
    env["gpu"] = torch.cuda.get_device_name(0)
    env["vram_total_mib"] = round(torch.cuda.mem_get_info()[1] / 1024**2)
(BASE / "configs" / "environment.json").write_text(json.dumps(env, indent=2), encoding="utf-8")
print(json.dumps(env, indent=2))

In [ ]:
#@title 5) Unduh data, audit duplikat, dan bentuk subset 2.000
import hashlib, json, re, unicodedata
import pandas as pd
from sklearn.model_selection import train_test_split

def norm(s):
    s = unicodedata.normalize("NFKC", str(s))
    return re.sub(r"\s+", " ", s).strip()

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW = BASE / "data" / "raw"
OUTD = BASE / "data" / "processed"
lm = CONFIG["label_map"]

for nama in CONFIG["data_files"].values():
    tujuan = RAW / nama
    if not tujuan.exists():
        url = f'{CONFIG["data_base_url"]}/{nama}'
        import urllib.request
        urllib.request.urlretrieve(url, tujuan)
        print("unduh:", nama, tujuan.stat().st_size, "byte")
    else:
        print("sudah ada:", nama)

def baca(fn):
    d = pd.read_csv(RAW / fn, sep="\t", header=None, names=["text", "label_name"],
                    quoting=3, keep_default_na=False)
    d["id"] = range(len(d))
    d["label"] = d["label_name"].map(lm)
    assert d["label"].notna().all(), "ada label tak dikenal"
    d["label"] = d["label"].astype(int)
    d["key"] = d["text"].map(norm)
    return d

tr, va, te = (baca(CONFIG["data_files"][k]) for k in ("train", "valid", "test"))
assert (len(tr), len(va), len(te)) == (11000, 1260, 500), "pembagian resmi tidak sesuai"

rep = {"awal": {"train": len(tr), "valid": len(va), "test": len(te)},
       "label_map": lm,
       "distribusi_awal": {k: d.label_name.value_counts().to_dict()
                           for k, d in (("train", tr), ("valid", va), ("test", te))}}
rep["duplikat_dalam"] = {k: int(d.key.duplicated().sum())
                         for k, d in (("train", tr), ("valid", va), ("test", te))}
rep["grup_duplikat_latih_label_berbeda"] = int((tr.groupby("key").label.nunique() > 1).sum())

c = tr.drop_duplicates("key", keep="first")
rep["latih_dihapus_duplikat_dalam"] = len(tr) - len(c)
ke = set(va.key) | set(te.key)
sebelum = len(c); c = c[~c.key.isin(ke)]
rep["latih_dihapus_overlap_valid_test"] = sebelum - len(c)
sebelum = len(va); va2 = va[~va.key.isin(set(te.key))]
rep["valid_dihapus_overlap_test"] = sebelum - len(va2)
rep["kandidat_latih_bersih"] = len(c)
rep["akhir_valid"], rep["akhir_test"] = len(va2), len(te)

n = CONFIG["n_train_subset"]
assert len(c) >= n, f"kandidat bersih {len(c)} < {n}"
idx, _ = train_test_split(c.id.values, train_size=n, stratify=c.label.values,
                          random_state=CONFIG["subset_seed"])
sub = c[c.id.isin(set(idx))].sort_values("id")
rep["subset_latih"] = len(sub)
rep["distribusi_subset"] = sub.label_name.value_counts().to_dict()

kolom = ["id", "text", "label", "label_name"]
sub[kolom].to_csv(OUTD / "train.csv", index=False)
va2[kolom].to_csv(OUTD / "valid.csv", index=False)
te[kolom].to_csv(OUTD / "test.csv", index=False)
daftar = sorted(int(i) for i in sub.id)
(OUTD / "indeks_subset_2000.json").write_text(json.dumps(daftar), encoding="utf-8")
rep["hash_indeks_subset"] = hashlib.sha256(json.dumps(daftar).encode()).hexdigest()
rep["hash_berkas_olahan"] = {k: sha256_file(OUTD / f"{k}.csv") for k in ("train", "valid", "test")}
(OUTD / "laporan_duplikat.json").write_text(json.dumps(rep, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: rep[k] for k in ["subset_latih", "distribusi_subset", "akhir_valid",
                                       "akhir_test", "hash_indeks_subset"]}, indent=2, ensure_ascii=False))

In [ ]:
#@title 6) Unduh model dan catat revisinya
from huggingface_hub import HfApi, snapshot_download

info = HfApi().model_info(CONFIG["model_name"])
print("revisi:", info.sha)
jalur = snapshot_download(CONFIG["model_name"], allow_patterns=["*.json", "*.safetensors", "*.txt"])
print("tersimpan:", jalur)
(BASE / "configs" / "model_info.json").write_text(json.dumps({
    "model_name": CONFIG["model_name"], "revision": info.sha, "lisensi": "Apache 2.0",
    "sumber": f'https://huggingface.co/{CONFIG["model_name"]}', "lokasi_cache": str(jalur),
}, indent=2), encoding="utf-8")

In [ ]:
#@title 7) Cek tokenizer: pastikan kode A, B, C masing-masing satu token
import json
from transformers import AutoTokenizer

LETTERS = ["A", "B", "C"]
tok = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

ids = {c: tok.encode(c, add_special_tokens=False) for c in LETTERS}
assert all(len(v) == 1 for v in ids.values()), f"kode jawaban bukan satu token: {ids}"
print("kode A, B, C ->", [ids[c][0] for c in LETTERS])
print("im_end      ->", tok.convert_tokens_to_ids("<|im_end|>"))

contoh = sub.text.iloc[0]
msgs = [{"role": "user", "content": CONFIG["prompt"].format(teks=contoh)}]
teks_prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
print("--- contoh prompt ---")
print(teks_prompt[:400])
(BASE / "logs" / "cek_tokenizer.json").write_text(json.dumps({
    "kode_jawaban": {c: ids[c][0] for c in LETTERS},
    "im_end": tok.convert_tokens_to_ids("<|im_end|>"),
    "pad": tok.pad_token, "eos": tok.eos_token,
    "contoh_prompt": teks_prompt,
}, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
#@title 8) Fungsi inti: penyiapan prompt, pelatihan, prediksi, metrik
import json, math, os, random, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix, f1_score
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, get_linear_schedule_with_warmup
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import bitsandbytes as bnb

LETTERS = ["A", "B", "C"]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def load_split(nama):
    return pd.read_csv(BASE / "data" / "processed" / f"{nama}.csv")

def tokenizer_baru():
    t = AutoTokenizer.from_pretrained(CONFIG["model_name"])
    if t.pad_token is None:
        t.pad_token = t.eos_token
    return t

def id_jawaban(t):
    return [t.encode(c, add_special_tokens=False)[0] for c in LETTERS]

def id_prompt(t, teks):
    msgs = [{"role": "user", "content": CONFIG["prompt"].format(teks=teks)}]
    s = t.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    return t(s, add_special_tokens=False)["input_ids"]

def prompt_muat(t, teks, max_len):
    '''Potong hanya bagian teks dari kanan agar prompt + jawaban + token akhir muat.'''
    cadangan = 2
    potong, terpotong = len(t(teks, add_special_tokens=False)["input_ids"]), False
    ids = id_prompt(t, teks)
    dasar = len(id_prompt(t, ""))
    while len(ids) + cadangan > max_len and potong > 0:
        potong = min(potong - 1, max(0, max_len - cadangan - dasar))
        terpotong = True
        ids = id_prompt(t, t.decode(t(teks, add_special_tokens=False)["input_ids"][:potong]))
    return ids, terpotong

def contoh_latih(t, teks, label, max_len):
    p, trunc = prompt_muat(t, teks, max_len)
    ans = [id_jawaban(t)[label], t.convert_tokens_to_ids("<|im_end|>")]
    return p + ans, [-100] * len(p) + ans, trunc

def model_dasar():
    bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_use_double_quant=True,
                                 bnb_4bit_compute_dtype=torch.float16)
    m = AutoModelForCausalLM.from_pretrained(CONFIG["model_name"], quantization_config=bnb_cfg,
                                             device_map={"": 0})
    m.config.use_cache = False
    return m

def prediksi(model, t, teks_daftar, max_len):
    '''Skor tiga kode jawaban pada posisi jawaban, prediksi = argmax. Tanpa generasi bebas.'''
    kode = id_jawaban(t)
    enc = [prompt_muat(t, x, max_len)[0] for x in teks_daftar]
    model.eval()
    keluaran = []
    bs = CONFIG["eval_batch"]
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(enc), bs):
            potong = enc[i:i + bs]
            L = max(len(x) for x in potong)
            ids_ = torch.tensor([[t.pad_token_id] * (L - len(x)) + x for x in potong]).cuda()
            att = torch.tensor([[0] * (L - len(x)) + [1] * len(x) for x in potong]).cuda()
            pos = (att.cumsum(-1) - 1).clamp(min=0)
            try:
                lg = model(input_ids=ids_, attention_mask=att, position_ids=pos,
                           logits_to_keep=1).logits[:, -1, :]
            except TypeError:
                lg = model(input_ids=ids_, attention_mask=att, position_ids=pos).logits[:, -1, :]
            keluaran.append(lg[:, kode].float().cpu().numpy())
    skor = np.concatenate(keluaran)
    return skor.argmax(1), skor

def metrik(y, p):
    return {
        "macro_f1": float(f1_score(y, p, labels=[0, 1, 2], average="macro", zero_division=0)),
        "f1_per_kelas": [float(v) for v in f1_score(y, p, labels=[0, 1, 2], average=None, zero_division=0)],
        "confusion_matrix": confusion_matrix(y, p, labels=[0, 1, 2]).tolist(),
    }

def macro_f1_cepat(y, p):
    cm = np.bincount(3 * y + p, minlength=9).reshape(3, 3)
    tp = np.diag(cm).astype(float)
    den = cm.sum(0) + cm.sum(1)
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return float(f1.mean())

print("fungsi inti siap")

In [ ]:
#@title 9) Fungsi pelatihan satu kondisi (rank, seed)
def latih(rank, seed, out_dir, pilot=False, max_len=None):
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    max_len = max_len or CONFIG["max_len"]
    res = {"rank": rank, "seed": seed, "max_len": max_len, "pilot": pilot,
           "status": "started", "model": CONFIG["model_name"], "waktu_mulai": time.strftime("%Y-%m-%d %H:%M:%S")}
    def simpan():
        (out / "hasil.json").write_text(json.dumps(res, indent=2, ensure_ascii=False), encoding="utf-8")

    try:
        set_seed(seed)
        t = tokenizer_baru()
        tr = load_split("train")
        ex, ntrunc = [], 0
        for teks, lab in zip(tr.text, tr.label):
            ids_, lab_, trunc = contoh_latih(t, teks, int(lab), max_len)
            ex.append((ids_, lab_)); ntrunc += trunc
        res["train_terpotong"] = ntrunc / len(ex)
        res["jumlah_train"] = len(ex)

        m = model_dasar()
        m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
        lora = LoraConfig(r=rank, lora_alpha=CONFIG["alpha_multiplier"] * rank,
                          lora_dropout=CONFIG["lora_dropout"], bias=CONFIG["bias"],
                          target_modules=CONFIG["target_modules"], task_type="CAUSAL_LM")
        m = get_peft_model(m, lora)
        params = [p for p in m.parameters() if p.requires_grad]
        res["parameter_terlatih"] = int(sum(p.numel() for p in params))
        res["alpha"] = lora.lora_alpha

        ga = CONFIG["grad_accum"]
        langkah_per_epoch = math.ceil(len(ex) / ga)
        total = CONFIG["epochs"] * langkah_per_epoch
        if pilot:
            total = CONFIG["pilot_steps"]
        res["langkah_dirancang"] = total

        opt = bnb.optim.AdamW8bit(params, lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
        sched = get_linear_schedule_with_warmup(opt, round(CONFIG["warmup_ratio"] * total), total)
        scaler = torch.amp.GradScaler("cuda")
        gen = torch.Generator().manual_seed(seed)

        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        res["memori_setelah_muat_mib"] = torch.cuda.memory_allocated() / 2**20
        t0 = time.perf_counter(); langkah = 0; loss_daftar = []
        m.train()
        for epoch in range(CONFIG["epochs"]):
            urutan = torch.randperm(len(ex), generator=gen).tolist()
            for g in range(0, len(urutan), ga):
                grup = urutan[g:g + ga]; gl = 0.0
                for i in grup:
                    ids_, lab_ = ex[i]
                    x = torch.tensor([ids_]).cuda(); y = torch.tensor([lab_]).cuda()
                    with torch.autocast("cuda", dtype=torch.float16):
                        loss = m(input_ids=x, labels=y).loss / len(grup)
                    scaler.scale(loss).backward(); gl += float(loss)
                if not math.isfinite(gl):
                    res["status"] = "loss_nan"; res["anomali"] = f"loss tidak finit pada langkah {langkah}"
                    simpan(); return res
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, CONFIG["max_grad_norm"])
                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)
                loss_daftar.append(gl); langkah += 1
                if langkah % 25 == 0:
                    print(f"  langkah {langkah}/{total} loss {gl:.4f}", flush=True)
                if langkah >= total:
                    break
            if langkah >= total:
                break

        torch.cuda.synchronize()
        res["waktu_pelatihan_detik"] = time.perf_counter() - t0
        res["peak_allocated_mib"] = torch.cuda.max_memory_allocated() / 2**20
        res["peak_reserved_mib"] = torch.cuda.max_memory_reserved() / 2**20
        res["langkah"] = langkah
        res["loss_akhir"] = loss_daftar[-1]
        res["loss_rata2_10_terakhir"] = float(np.mean(loss_daftar[-10:]))
        res["status"] = "ok"
        simpan()
        pd.DataFrame({"langkah": range(1, len(loss_daftar) + 1), "loss": loss_daftar}).to_csv(
            out / "kurva_loss.csv", index=False)

        if not pilot:
            m.save_pretrained(out / "adaptor")
            res["ukuran_adaptor_mib"] = sum(f.stat().st_size for f in (out / "adaptor").rglob("*")
                                            if f.is_file()) / 2**20
            va = load_split("valid")
            p, s = prediksi(m, t, list(va.text), max_len)
            res["validasi"] = metrik(va.label.values, p)
            res["macro_f1_validasi"] = res["validasi"]["macro_f1"]
            pd.DataFrame({"id": va.id, "label": va.label, "pred": p,
                          "sA": s[:, 0], "sB": s[:, 1], "sC": s[:, 2]}).to_csv(
                out / "pred_valid.csv", index=False)
            simpan()

        del m, opt, params
        torch.cuda.empty_cache()
        return res

    except torch.cuda.OutOfMemoryError:
        res["status"] = "oom"; res["anomali"] = "CUDA out of memory"
        simpan(); torch.cuda.empty_cache(); return res
    except Exception:
        res["status"] = "error"; res["anomali"] = traceback.format_exc()
        simpan(); return res

print("fungsi latih siap")

In [ ]:
#@title 10) Pilot: rank 32, 20 langkah, untuk menguji kelayakan memori
pilot_dir = BASE / "runs" / "pilot_r32_s42"
if (pilot_dir / "hasil.json").exists():
    print("pilot sudah ada, dilewati")
    print(json.loads((pilot_dir / "hasil.json").read_text()))
else:
    hasil_pilot = latih(32, 42, pilot_dir, pilot=True)
    print(json.dumps({k: v for k, v in hasil_pilot.items() if k != "anomali"}, indent=2, ensure_ascii=False))
    if hasil_pilot["status"] == "oom":
        print("\nPILOT OOM. Buka sel 4, ubah max_len menjadi 128, jalankan ulang dari sel 4, "
              "lalu catat perubahan itu pada notes/changelog.md.")

In [ ]:
#@title 11) Pelatihan utama: 4 rank x 3 seed, bisa dilanjutkan
import random, time

sched_f = BASE / "configs" / "urutan_eksekusi.json"
if sched_f.exists():
    schedule = json.loads(sched_f.read_text())
else:
    rng = random.Random(CONFIG["schedule_seed"]); schedule = []
    for s in CONFIG["seeds"]:
        rs = list(CONFIG["ranks"]); rng.shuffle(rs)
        schedule += [{"seed": s, "rank": r} for r in rs]
    sched_f.write_text(json.dumps(schedule, indent=2))
print("urutan eksekusi:", [(i["rank"], i["seed"]) for i in schedule])

for item in schedule:
    nama = f"r{item['rank']}_s{item['seed']}"
    out = BASE / "runs" / nama
    hf = out / "hasil.json"
    if hf.exists() and json.loads(hf.read_text()).get("status") == "ok" and (out / "pred_valid.csv").exists():
        print(f"lewati (sudah selesai): {nama}")
        continue
    print(f"\n=== mulai {nama} pada {time.strftime('%H:%M:%S')} ===", flush=True)
    t0 = time.time()
    r = latih(item["rank"], item["seed"], out)
    print(f"=== {nama} status={r['status']} dalam {(time.time()-t0)/60:.1f} menit ===", flush=True)
    if r["status"] != "ok":
        print("PERHATIAN: kondisi gagal. Jangan dihapus. Catat di notes/changelog.md, "
              "lalu ulangi dengan setelan yang sama.")
    torch.cuda.empty_cache()

selesai = [p.name for p in (BASE / "runs").glob("r*_s*/hasil.json")]
print("\nrun yang punya hasil.json:", sorted(selesai))

In [ ]:
#@title 12) Agregasi data validasi dan pemilihan rank
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

baris = []
for r in CONFIG["ranks"]:
    for s in CONFIG["seeds"]:
        f = BASE / "runs" / f"r{r}_s{s}" / "hasil.json"
        h = json.loads(f.read_text()) if f.exists() else {"status": "belum"}
        va = h.get("validasi") or {}
        baris.append({
            "rank": r, "seed": s, "status": h.get("status"),
            "macro_f1_validasi": h.get("macro_f1_validasi"),
            "f1_negatif": (va.get("f1_per_kelas") or [None]*3)[0],
            "f1_netral": (va.get("f1_per_kelas") or [None]*3)[1],
            "f1_positif": (va.get("f1_per_kelas") or [None]*3)[2],
            "confusion_matrix": va.get("confusion_matrix"),
            "parameter_terlatih": h.get("parameter_terlatih"),
            "peak_allocated_mib": h.get("peak_allocated_mib"),
            "peak_reserved_mib": h.get("peak_reserved_mib"),
            "waktu_pelatihan_detik": h.get("waktu_pelatihan_detik"),
            "ukuran_adaptor_mib": h.get("ukuran_adaptor_mib"),
            "loss_akhir": h.get("loss_akhir"),
            "train_terpotong": h.get("train_terpotong"),
        })
df = pd.DataFrame(baris)
df.to_csv(BASE / "results" / "semua_run_validasi.csv", index=False)

agg = []
for rank, g in df.groupby("rank"):
    ok = g[g.status == "ok"]
    d = {"rank": int(rank), "jumlah_run_ok": int(len(ok))}
    for c in ["macro_f1_validasi", "f1_negatif", "f1_netral", "f1_positif",
              "peak_allocated_mib", "peak_reserved_mib", "waktu_pelatihan_detik",
              "parameter_terlatih", "ukuran_adaptor_mib"]:
        v = pd.to_numeric(ok[c], errors="coerce")
        d[c + "_mean"] = float(v.mean()) if v.notna().any() else None
        d[c + "_sd"] = float(v.std(ddof=1)) if v.notna().sum() > 1 else None
    agg.append(d)
agg = pd.DataFrame(agg).sort_values("rank")
agg["selisih_dari_terbaik"] = agg.macro_f1_validasi_mean.max() - agg.macro_f1_validasi_mean
agg.to_csv(BASE / "results" / "tabel_rank_validasi.csv", index=False)
print(agg.round(4).to_string(index=False))

layak = agg[(agg.jumlah_run_ok > 0) & agg.macro_f1_validasi_mean.notna()]
if len(layak) == 0:
    print("\nBelum ada run selesai. Jalankan sel 11 sampai selesai.")
else:
    terbaik = float(layak.macro_f1_validasi_mean.max())
    kandidat = layak[layak.macro_f1_validasi_mean >= terbaik - CONFIG["tolerance"] - 1e-12]
    pilih = int(kandidat["rank"].min())
    rec = {"rank_terpilih": pilih, "macro_f1_validasi_terbaik": terbaik,
           "toleransi": CONFIG["tolerance"], "dibekukan_pada": time.strftime("%Y-%m-%d %H:%M:%S"),
           "jumlah_seed_per_rank": {int(r): int(n) for r, n in zip(agg["rank"], agg["jumlah_run_ok"])}}
    (BASE / "results" / "rank_terpilih.json").write_text(json.dumps(rec, indent=2), encoding="utf-8")
    print("\nRank terpilih (dibekukan):", rec)

    TINTA = "#111827"
    fig, axs = plt.subplots(1, 3, figsize=(11.5, 3.3), dpi=200)
    for ax, (kol, label, warna) in zip(axs, [
        ("macro_f1_validasi", "Macro-F1 validasi", "#1d4ed8"),
        ("peak_reserved_mib", "Puncak memori reserved (MiB)", "#b45309"),
        ("waktu_pelatihan_detik", "Waktu pelatihan (detik)", "#15803d")]):
        ax.errorbar(agg["rank"], agg[kol + "_mean"], yerr=agg[kol + "_sd"].fillna(0),
                    marker="o", color=warna, capsize=3, linewidth=1.6, markersize=5)
        ax.set_xscale("log", base=2); ax.set_xticks(list(agg["rank"]))
        ax.set_xticklabels([f"r={int(r)}" for r in agg["rank"]], fontsize=8.5)
        ax.set_xlabel("Rank adaptor", fontsize=9); ax.set_ylabel(label, fontsize=9)
        ax.grid(alpha=0.25, linewidth=0.6); ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Kinerja dan biaya komputasi menurut rank adaptor (data validasi)", fontsize=10.5)
    fig.tight_layout()
    fig.savefig(BASE / "results" / "grafik_rank.png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("gambar disimpan: results/grafik_rank.png")

In [ ]:
#@title 13) Evaluasi data uji SEKALI (dijalankan setelah rank dibekukan)
from peft import PeftModel

fz = BASE / "results" / "rank_terpilih.json"
if not fz.exists():
    raise SystemExit("Rank belum dibekukan. Selesaikan sel 12 lebih dulu.")
if (BASE / "results" / "hasil_uji.json").exists():
    print("hasil_uji.json sudah ada; data uji tidak dievaluasi ulang.")
    print((BASE / "results" / "hasil_uji.json").read_text()[:600])
else:
    r = json.loads(fz.read_text())["rank_terpilih"]
    te = load_split("test"); y = te.label.values
    t = tokenizer_baru()
    berkas_maxlen = BASE / "runs" / f"r{r}_s{CONFIG['seeds'][0]}" / "hasil.json"
    max_len = (json.loads(berkas_maxlen.read_text())["max_len"]
               if berkas_maxlen.exists() else CONFIG["max_len"])

    preds = {}
    dasar = model_dasar()
    preds["zero_shot"] = prediksi(dasar, t, list(te.text), max_len)
    del dasar; torch.cuda.empty_cache()

    for s in CONFIG["seeds"]:
        jalur_ad = BASE / "runs" / f"r{r}_s{s}" / "adaptor"
        if not jalur_ad.exists():
            print(f"adaptor seed {s} tidak ada, dilewati")
            continue
        m = PeftModel.from_pretrained(model_dasar(), str(jalur_ad))
        preds[f"seed{s}"] = prediksi(m, t, list(te.text), max_len)
        del m; torch.cuda.empty_cache()

    out = {"rank": r, "max_len": max_len, "metrik": {}}
    kolom = {"id": te.id, "label": y}
    for k, (p, sc) in preds.items():
        out["metrik"][k] = metrik(y, p)
        kolom[f"pred_{k}"] = p
        assert abs(out["metrik"][k]["macro_f1"] - macro_f1_cepat(y, p)) < 1e-9

    tr = load_split("train"); mayor = int(tr.label.value_counts().idxmax())
    out["metrik"]["kelas_mayoritas"] = metrik(y, np.full(len(y), mayor))
    out["kelas_mayoritas"] = mayor
    pd.DataFrame(kolom).to_csv(BASE / "results" / "pred_uji.csv", index=False)

    rng = np.random.default_rng(CONFIG["bootstrap_seed"])
    kidx = [np.where(y == k)[0] for k in (0, 1, 2)]
    P = [preds[k][0] for k in preds if k.startswith("seed")]
    Z = preds["zero_shot"][0]
    diff = []
    for _ in range(CONFIG["bootstrap_reps"]):
        idx = np.concatenate([rng.choice(v, size=len(v), replace=True) for v in kidx])
        diff.append(np.mean([macro_f1_cepat(y[idx], p[idx]) for p in P]) - macro_f1_cepat(y[idx], Z[idx]))
    lo, hi = np.percentile(diff, [2.5, 97.5])
    f_seed = [out["metrik"][k]["macro_f1"] for k in out["metrik"] if k.startswith("seed")]
    out["ringkasan"] = {
        "macro_f1_uji_per_seed": f_seed,
        "mean": float(np.mean(f_seed)) if f_seed else None,
        "sd": float(np.std(f_seed, ddof=1)) if len(f_seed) > 1 else None,
        "macro_f1_zero_shot": out["metrik"]["zero_shot"]["macro_f1"],
        "macro_f1_kelas_mayoritas": out["metrik"]["kelas_mayoritas"]["macro_f1"],
        "selisih_mean_terhadap_zero_shot": float(np.mean(f_seed) - out["metrik"]["zero_shot"]["macro_f1"]) if f_seed else None,
        "bootstrap_persentil_2_5": float(lo), "bootstrap_persentil_97_5": float(hi),
        "replikasi": CONFIG["bootstrap_reps"],
    }
    (BASE / "results" / "hasil_uji.json").write_text(json.dumps(out, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(out["ringkasan"], indent=2, ensure_ascii=False))

In [ ]:
#@title 14) Paketkan seluruh hasil menjadi satu zip untuk dikirim
import shutil, zipfile

ZIP = Path("/content/hasil-colab.zip")
if ZIP.exists():
    ZIP.unlink()

ringkas = BASE / "results" / "RINGKASAN.md"
baris = ["# Ringkasan hasil eksperimen (Colab)", ""]
env = json.loads((BASE / "configs" / "environment.json").read_text())
baris += [f"- GPU: {env.get('gpu')} | VRAM {env.get('vram_total_mib')} MiB | torch {env.get('torch')} | CUDA {env.get('cuda')}",
          f"- transformers {env['packages'].get('transformers')} | peft {env['packages'].get('peft')} | bitsandbytes {env['packages'].get('bitsandbytes')}",
          ""]
tbl = BASE / "results" / "tabel_rank_validasi.csv"
if tbl.exists():
    baris += ["## Tabel validasi per rank", "", "```",
              pd.read_csv(tbl).round(4).to_string(index=False), "```", ""]
if (BASE / "results" / "rank_terpilih.json").exists():
    baris += ["## Rank terpilih", "", "```",
              (BASE / "results" / "rank_terpilih.json").read_text(), "```", ""]
if (BASE / "results" / "hasil_uji.json").exists():
    hu = json.loads((BASE / "results" / "hasil_uji.json").read_text())
    baris += ["## Hasil uji akhir", "", "```",
              json.dumps(hu["ringkasan"], indent=2, ensure_ascii=False), "```", ""]
ringkas.write_text("\n".join(baris), encoding="utf-8")

with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for sub in ["configs", "data/processed", "runs", "results", "logs", "notes"]:
        for f in (BASE / sub).rglob("*"):
            if f.is_file() and not f.name.endswith((".safetensors", ".bin", ".pt")):
                z.write(f, f"{sub}/{f.relative_to(BASE / sub)}")
    z.write(ringkas, "results/RINGKASAN.md")

print(f"zip dibuat: {ZIP} ({ZIP.stat().st_size/1024/1024:.1f} MB)")
print("unduh dari panel Files di kiri, lalu kirim berkas ini untuk diolah menjadi laporan.")
print("\nIsi ringkas:")
print(ringkas.read_text()[:1500])

## Catatan dan penanganan masalah

**Sesi terputus.** Sambungkan ulang Runtime, jalankan sel 1 sampai 4, lalu langsung sel 11.
Run yang sudah selesai dilewati karena `hasil.json` dan `pred_valid.csv` sudah ada di Drive.

**Pilot kehabisan memori.** T4 punya 15 GB sehingga konfigurasi 256 token seharusnya muat.
Bila tetap OOM, buka sel 4, ubah `max_len` menjadi 128, jalankan ulang sel 4, lalu sel 11.
Perubahan itu wajib dicatat di `notes/changelog.md` agar dilaporkan jujur di laporan.

**Waktu.** Pada GPU T4, satu run 375 langkah biasanya 10 sampai 20 menit. Dua belas run
berarti sekitar 3 sampai 4 jam. Jalankan pada sesi yang cukup panjang, dan biarkan tab tetap aktif.

**Urutan yang boleh diubah.** Jangan mengubah `ranks`, `seeds`, `tolerance`, `alpha_multiplier`,
`target_modules`, atau prompt setelah pelatihan dimulai. Mengubahnya di tengah jalan membuat
perbandingan tidak sah. Semua perubahan harus dicatat di `notes/changelog.md`.

**Data uji.** Sel 13 hanya jalan sekali. Jika `hasil_uji.json` sudah ada, sel itu menolak
mengevaluasi ulang supaya angka uji tidak dipakai untuk memilih konfigurasi.

**Berkas yang tidak masuk zip.** Bobot adaptor (`*.safetensors`) tidak dipaketkan agar zip
kecil. Bila adaptor juga dibutuhkan, salin manual folder `runs/r*_s*/adaptor/`.